# Shape analysis

In [ ]:
import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import os
from geopandas import GeoDataFrame
from matplotlib.patches import Patch
from matplotlib_scalebar.scalebar import ScaleBar
from pandas import DataFrame

%config InlineBackend.figure_format = 'retina'

gdf = gpd.read_file('../../../../input/v6.4/umweltzone/Umweltzone_Berlin.shp')
bezirke = gpd.read_file("/Users/paulh/git/matsim-berlin/input/v6.4/bezirksgrenzen.geojson")

area_inner = gdf.to_crs(epsg=6933).area.sum() / 1e6
print(f"Area inner: {area_inner:.2f} km²")
area_berlin = bezirke.to_crs(epsg=6933).area.sum() / 1e6
print(f"Area Berlin: {area_berlin:.2f} km²")

gdf = gdf.to_crs(epsg=3857)
bezirke = bezirke.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(12, 12))

# Plot layers (labels here won't be used by legend; we add proxies below)
bezirke.plot(
    ax=ax,
    facecolor="red",
    # edgecolor="black",
    # edgecolor="red",
    # linewidth=1.0,
    alpha=0.4,
)

gdf.plot(
    ax=ax,
    facecolor="blue",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.4,
)

ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, zoom=12)

scalebar = ScaleBar(dx=1, units="m", location="lower right", scale_loc="bottom")
ax.add_artist(scalebar)

# --- Manual legend with proxy artists ---
legend_handles = [
    Patch(facecolor="red", edgecolor="red", alpha=0.4, linewidth=1.0, label="City of Berlin"),
    Patch(facecolor="blue", edgecolor="black", linewidth=0.5, alpha=0.5, label='Inner Ring ("Car-Free Area")'),
]
ax.legend(handles=legend_handles, loc="upper right", frameon=True)

ax.set_axis_off()
# ax.set_title('Berlin', fontsize=18)
os.makedirs("./plots", exist_ok=True)
plt.savefig("./plots/" + "berlin.pdf", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()

# Link Stat Analysis

In [ ]:


pct = 0.01


def read_links(input_path) -> GeoDataFrame:
    import pandas as pd

    df_links = pd.read_csv(
        input_path,
        sep=";",
        compression="zstd",
        dtype={"link": "string"},
        low_memory=False,
    )

    links = gpd.GeoDataFrame(
        df_links[~df_links["link"].str.contains("pt_")],
        geometry=gpd.GeoSeries.from_wkt(df_links["geometry"], crs="EPSG:25832"),
    )

    pcu_factors = {
        "vol_car": 1,
        "vol_freight": 3.5,
        "vol_truck": 3.5,
        "vol_commercial_goods_car": 1.0
    }

    links["vol_scaled"] = (
            sum(links.get(col, 0) * factor for col, factor in pcu_factors.items()) / pct
    )
    return links


def plot_link_volumes(links, max_vol, zone, title="Daily Link Volumes"):
    links_3857 = links.to_crs(epsg=3857)
    links_3857["geometry"] = links_3857.geometry.simplify(2)

    bezirke_union = bezirke.geometry.union_all()
    links_3857 = links_3857[links_3857.intersects(bezirke_union)]

    # Thickness ~ capacity (clipped to robust quantiles for readability)
    cap = links_3857["capacity"].astype(float)
    cap_q_low, cap_q_high = cap.quantile([0.02, 0.98])
    cap_clip = cap.clip(lower=cap_q_low, upper=cap_q_high)
    # normalize capacity value to 0.0 - 1.0
    cap_norm = (cap_clip - cap_q_low) / (cap_q_high - cap_q_low) if cap_q_high > cap_q_low else cap_clip * 0 + 0.5
    links_3857["_lw"] = 0.2 + cap_norm * 3.0  # linewidth range ~ [0.2, 3.2]

    vmax_in = float(max_vol.max()) if hasattr(max_vol, "max") else float(max_vol)

    fig, ax = plt.subplots(figsize=(12, 12))

    from matplotlib.colors import TwoSlopeNorm, Normalize

    vmin_data = float(links_3857["vol_scaled"].min())
    vmax_data = float(links_3857["vol_scaled"].max())
    vmax = max(vmax_in, vmax_data)

    if vmin_data < 0 < vmax_data:
        norm = TwoSlopeNorm(
            vmin=vmin_data,
            vcenter=0,
            vmax=vmax,
        )
        cmap = "bwr"
    else:
        # vol_scaled is non-negative (or does not cross zero): scale from 0..max_vol
        vmax_nonneg = vmax if vmax > 0 else 1.0
        norm = Normalize(vmin=0.0, vmax=vmax_nonneg)
        cmap = "viridis"

    links_3857.plot(
        ax=ax,
        column="vol_scaled",
        cmap=cmap,
        norm=norm,
        linewidth=links_3857["_lw"],
        alpha=0.8,
        legend=True,
        legend_kwds={"label": "Traffic Volume in Passenger Car Equivalents", "shrink": 0.6},
    )

    zone.plot(
        ax=ax,
        facecolor="orange",
        edgecolor="black",
        linewidth=0.5,
        alpha=0.2,
    )

    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=12)
    ax.set_axis_off()

    ax.set_title(title, fontsize=16)

    # adjust the centering of the map
    minx, miny, maxx, maxy = bezirke.total_bounds

    x_diff = maxx - minx
    y_diff = maxy - miny
    buffer_factor = -0.3
    minx -= x_diff * buffer_factor * 0.8
    maxx += x_diff * buffer_factor * 1.2
    miny -= y_diff * buffer_factor
    maxy += y_diff * buffer_factor

    ax.set_xlim([minx, maxx])
    ax.set_ylim([miny, maxy])

    plt.savefig("./plots/" + title + ".pdf", dpi=300, bbox_inches="tight", pad_inches=0)
    plt.show()


In [ ]:
base_path = "/Users/paulh/runs-svn/matsim-berlin/autofrei/1pct-v6.4/berlin-autofrei-v6.4-baseCaseCtdExtended/berlin-v6.4.output_links.csv.zst"
base_links = read_links(base_path)

policy_path = "/Users/paulh/runs-svn/matsim-berlin/autofrei/1pct-v6.4/berlin-autofrei-v6.4-policy/berlin-v6.4.output_links.csv.zst"
policy_links = read_links(policy_path)

max_vol = max(base_links["vol_scaled"].max(), policy_links["vol_scaled"].max())

In [ ]:
plot_link_volumes(base_links, max_vol, gdf, title="Daily Link Volumes (Base Case)")

In [ ]:
plot_link_volumes(policy_links, max_vol, gdf, title="Daily Link Volumes (Policy Case)")


In [ ]:
diff_links = (
    base_links[["link", "geometry", "vol_scaled", "capacity", "length"]]
    .rename(columns={"vol_scaled": "vol_scaled_base"})
    .merge(
        policy_links[["link", "vol_scaled"]].rename(columns={"vol_scaled": "vol_scaled_policy"}),
        on="link",
        how="inner",
    )
)

diff_links["vol_scaled"] = diff_links["vol_scaled_policy"] - diff_links["vol_scaled_base"]
diff_links = gpd.GeoDataFrame(diff_links, geometry="geometry", crs=base_links.crs)

plot_link_volumes(diff_links, diff_links["vol_scaled"].max(), gdf,
                  title="Difference in Daily Link Volumes (Policy - Base)")

In [ ]:
_zone = gdf.to_crs(diff_links.crs) if diff_links.crs != gdf.crs else gdf
_zone_union = _zone.geometry.union_all()
diff_links_inner = diff_links[diff_links.intersects(_zone_union)].copy()
diff_links_inner["veh_km_base"] = diff_links_inner["vol_scaled_base"] * diff_links_inner["length"] / 1000
diff_links_inner["veh_km_policy"] = diff_links_inner["vol_scaled_policy"] * diff_links_inner["length"] / 1000
# diff_links_inner

# sum the veh_km_base and veh_km_policy and print them:
veh_km_base_total = float(diff_links_inner["veh_km_base"].sum())
veh_km_policy_total = float(diff_links_inner["veh_km_policy"].sum())
veh_km_delta_total = veh_km_policy_total - veh_km_base_total
veh_km_delta_pct = (veh_km_delta_total / veh_km_base_total) if veh_km_base_total != 0 else float("nan")

print(f"Inner ring veh-km (base):   {veh_km_base_total:,.1f}")
print(f"Inner ring veh-km (policy): {veh_km_policy_total:,.1f}")
print(f"Δ (policy - base):          {veh_km_delta_total:,.1f} ({veh_km_delta_pct:+.2%})")


In [ ]:
_zone = gdf.to_crs(diff_links.crs)
_bezirke = bezirke.to_crs(diff_links.crs)

out = gpd.overlay(_bezirke, _zone, how="difference")

diff_links_out = diff_links[diff_links.intersects(out.union_all())].copy()
diff_links_out["veh_km_base"] = diff_links_out["vol_scaled_base"] * diff_links_out["length"] / 1000
diff_links_out["veh_km_policy"] = diff_links_out["vol_scaled_policy"] * diff_links_out["length"] / 1000
# diff_links_inner

# sum the veh_km_base and veh_km_policy and print them:
veh_km_base_total = float(diff_links_out["veh_km_base"].sum())
veh_km_policy_total = float(diff_links_out["veh_km_policy"].sum())
veh_km_delta_total = veh_km_policy_total - veh_km_base_total
veh_km_delta_pct = (veh_km_delta_total / veh_km_base_total) if veh_km_base_total != 0 else float("nan")

print(f"Outer ring veh-km (base):   {veh_km_base_total:,.1f}")
print(f"Outer ring veh-km (policy): {veh_km_policy_total:,.1f}")
print(f"Δ (policy - base):          {veh_km_delta_total:,.1f} ({veh_km_delta_pct:+.2%})")

In [ ]:
pl = policy_links.to_crs(epsg=3857).copy()
pl["geometry"] = pl.geometry.simplify(2)

bezirke_union = bezirke.geometry.union_all()
pl = pl[pl.intersects(bezirke_union)]

modes = pl["modes"].astype("string").fillna("")
has_car = (
        modes.str.contains(",car,", regex=False)
        | modes.str.contains("car,c", regex=False)
        | modes.str.startswith("car,", na=False)
        | modes.str.endswith(",car", na=False)
        | (modes == "car")
)

pl_car = pl[has_car].copy()
pl_nocar = pl[~has_car].copy()

# linewidth from capacity (robustly clipped)
cap_all = pl["capacity"].astype(float)
cap_q_low, cap_q_high = cap_all.quantile([0.02, 0.98])
cap_clip = cap_all.clip(lower=cap_q_low, upper=cap_q_high)
cap_norm = (cap_clip - cap_q_low) / (cap_q_high - cap_q_low) if cap_q_high > cap_q_low else cap_clip * 0 + 0.5
pl["_lw"] = 0.2 + cap_norm * 3.0

pl_car["_lw"] = pl.loc[pl_car.index, "_lw"]
pl_nocar["_lw"] = pl.loc[pl_nocar.index, "_lw"]

fig, ax = plt.subplots(figsize=(12, 12))

if len(pl_nocar) > 0:
    pl_nocar.plot(
        ax=ax,
        # column="vol_scaled",
        # cmap=cmap,
        # norm=norm,
        linewidth=pl_nocar["_lw"],
        alpha=0.1,
    )

pl_car.plot(
    ax=ax,
    # column="vol_scaled",
    # cmap=cmap,
    # norm=norm,
    linewidth=pl_car["_lw"],
    alpha=1.0,
    legend=True,
    legend_kwds={"label": "Traffic Volume in Traffic Volume in Passenger Car Equivalents", "shrink": 0.6},
)

gdf.to_crs(epsg=3857).plot(
    ax=ax,
    facecolor="green",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.2,
)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=12)
ax.set_axis_off()
# ax.set_title('Car Links', fontsize=16)

minx, miny, maxx, maxy = bezirke.to_crs(epsg=3857).total_bounds
x_diff, y_diff = (maxx - minx), (maxy - miny)
buffer_factor = -0.3
minx -= x_diff * buffer_factor * 0.8
maxx += x_diff * buffer_factor * 1.2
miny -= y_diff * buffer_factor
maxy += y_diff * buffer_factor
ax.set_xlim([minx, maxx])
ax.set_ylim([miny, maxy])

plt.savefig("./plots/" + "berlin-policy-links.pdf", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()


# Trip Analysis

In [ ]:

from geopandas import GeoDataFrame
import pandas as pd


def read_trips(input_folder) -> pd.DataFrame:
    df = pd.read_csv(
        input_folder + "/berlin-v6.4.output_trips.csv.zst",
        sep=";",
        compression="zstd",
        dtype={"person": "string"},
        low_memory=False,
    )

    home_locations = df.loc[df["trip_number"] == 1, ["person", "start_x", "start_y"]]
    home_locations["home_x"] = home_locations["start_x"]
    home_locations["home_y"] = home_locations["start_y"]
    home_locations = home_locations.drop(columns=["start_x", "start_y"])
    df = df.merge(home_locations, on="person", how="left")
    df = df[df["person"].str.startswith("berlin_", na=False)]
    return df


def filter_trips_by_zone(trips, zone: GeoDataFrame, invert=False) -> GeoDataFrame:
    trips_crs = "EPSG:25832"

    pts = gpd.points_from_xy(trips["home_x"], trips["home_y"])
    trips_gdf = gpd.GeoDataFrame(trips.copy(), geometry=pts, crs=trips_crs)

    zone_gdf = zone
    if zone_gdf.crs is None:
        raise ValueError("zone GeoDataFrame must have a CRS set")

    if trips_gdf.crs != zone_gdf.crs:
        trips_gdf = trips_gdf.to_crs(zone_gdf.crs)

    zone_union = zone_gdf.geometry.union_all()

    if invert:
        mask = ~trips_gdf.geometry.within(zone_union)
    else:
        mask = trips_gdf.geometry.within(zone_union)
    mask = mask.fillna(False) if hasattr(mask, "fillna") else np.asarray(mask, dtype=bool)

    return trips_gdf.loc[mask].copy()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns


def mode_share(trips_gdf: pd.DataFrame) -> pd.Series:
    counts = (
        trips_gdf["main_mode"]
        .astype("string")
        .fillna("unknown")
        .value_counts(dropna=False)
    )
    share = counts / counts.sum()
    return share


def mode_share_absolute(trips_gdf: pd.DataFrame) -> pd.Series:
    counts = (
        trips_gdf["main_mode"]
        .astype("string")
        .fillna("unknown")
        .value_counts(dropna=False)
    )
    counts_scaled = counts / pct
    return counts_scaled


def plot_modal_splits(base, policy, location):
    sns.set_theme(style="whitegrid")

    base_share = mode_share(base)
    policy_share = mode_share(policy)

    # consistent mode order across both (sorted by avg share for nicer reading)
    all_modes = sorted(set(base_share.index).union(set(policy_share.index)))
    avg_share = pd.Series(
        {m: (base_share.get(m, 0.0) + policy_share.get(m, 0.0)) / 2.0 for m in all_modes}
    )
    modes_order = avg_share.index.tolist()

    base_vals = np.array([float(base_share.get(m, 0.0)) for m in modes_order])
    policy_vals = np.array([float(policy_share.get(m, 0.0)) for m in modes_order])

    x = np.arange(len(modes_order))
    bar_w = 0.42

    fig, ax = plt.subplots(figsize=(10, 3.6))

    ax.bar(
        x - bar_w / 2,
        base_vals,
        width=bar_w,
        label="Base",
        color=sns.color_palette("Set2", 2)[0],
        edgecolor="white",
        linewidth=1.0,
    )
    ax.bar(
        x + bar_w / 2,
        policy_vals,
        width=bar_w,
        label="Policy",
        color=sns.color_palette("Set2", 2)[1],
        edgecolor="white",
        linewidth=1.0,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(modes_order)
    ax.set_ylim(0, 0.5)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("Mode")
    ax.set_ylabel("Share of trips")
    title = "Main Modal Split (" + location + "): Base vs Policy"
    ax.set_title(title)
    ax.legend(frameon=False)

    sns.despine(left=False, bottom=False)
    plt.tight_layout()

    plt.savefig("./plots/" + title + ".pdf", dpi=300, bbox_inches="tight", pad_inches=0)
    plt.show()


In [ ]:
input_folder_policy = "/Users/paulh/runs-svn/matsim-berlin/autofrei/1pct-v6.4/berlin-autofrei-v6.4-policy"
policy_trips = read_trips(input_folder_policy)

input_folder_base = "/Users/paulh/runs-svn/matsim-berlin/autofrei/1pct-v6.4/berlin-autofrei-v6.4-baseCaseCtdExtended"
base_trips = read_trips(input_folder_base)

policy_trips_in_zone = filter_trips_by_zone(policy_trips, gdf)
base_trips_in_zone = filter_trips_by_zone(base_trips, gdf)

plot_modal_splits(base_trips_in_zone, policy_trips_in_zone, "Inner Ring")

In [ ]:
policy_trips_out_zone = filter_trips_by_zone(policy_trips, gdf, invert=True)
base_trips_out_zone = filter_trips_by_zone(base_trips, gdf, invert=True)

plot_modal_splits(base_trips_out_zone, policy_trips_out_zone, "Out of Inner Ring")

In [ ]:
plot_modal_splits(base_trips, policy_trips, "Berlin")

In [ ]:
base_share = mode_share(base_trips)
policy_share = mode_share(policy_trips)

base_share_in = mode_share(base_trips_in_zone)
policy_share_in = mode_share(policy_trips_in_zone)

base_share_out = mode_share(base_trips_out_zone)
policy_share_out = mode_share(policy_trips_out_zone)

all_modes = sorted(
    set(base_share.index)
    .union(set(policy_share.index))
    .union(set(base_share_in.index))
    .union(set(policy_share_in.index))
    .union(set(base_share_out.index))
    .union(set(policy_share_out.index))
)

mode_share_table = (
    pd.DataFrame(
        {
            "Mode": all_modes,
            "B (Berlin)": [float(base_share.get(m, 0.0)) for m in all_modes],
            "P (Berlin)": [float(policy_share.get(m, 0.0)) for m in all_modes],
            "B (IR)": [float(base_share_in.get(m, 0.0)) for m in all_modes],
            "P (IR)": [float(policy_share_in.get(m, 0.0)) for m in all_modes],
            "B (Out)": [float(base_share_out.get(m, 0.0)) for m in all_modes],
            "P (Out)": [float(policy_share_out.get(m, 0.0)) for m in all_modes],
        }
    )
    # .sort_values(["Base (Berlin)", "Policy (Berlin)"], ascending=False)
    .reset_index(drop=True)
)

latex_table = mode_share_table.to_latex(
    index=False,
    column_format="lcccccc",
    float_format=lambda x: f"{x:.1%}".replace("%", r"\%"),
)

# latex_table = (
#     latex_table.replace(r"\toprule", r"\hline", 1)
#     .replace(r"\midrule", r"\hline", 1)
#     .replace(r"\bottomrule", r"\hline", 1)
# )

print(latex_table)


In [ ]:
policy_share

In [ ]:
base_share_abs = mode_share_absolute(base_trips)
policy_share_abs = mode_share_absolute(policy_trips)

base_share_in_abs = mode_share_absolute(base_trips_in_zone)
policy_share_in_abs = mode_share_absolute(policy_trips_in_zone)

base_share_out_abs = mode_share_absolute(base_trips_out_zone)
policy_share_out_abs = mode_share_absolute(policy_trips_out_zone)

all_modes_abs = sorted(
    set(base_share_abs.index)
    .union(set(policy_share_abs.index))
    .union(set(base_share_in_abs.index))
    .union(set(policy_share_in_abs.index))
    .union(set(base_share_out_abs.index))
    .union(set(policy_share_out_abs.index))
)

mode_share_table = (
    pd.DataFrame(
        {
            "Mode": all_modes_abs,
            "B (Berlin)": [float(base_share_abs.get(m, 0.0)) for m in all_modes_abs],
            "P (Berlin)": [float(policy_share_abs.get(m, 0.0)) for m in all_modes_abs],
            "B (IR)": [float(base_share_in_abs.get(m, 0.0)) for m in all_modes_abs],
            "P (IR)": [float(policy_share_in_abs.get(m, 0.0)) for m in all_modes_abs],
            "B (OR)": [float(base_share_out_abs.get(m, 0.0)) for m in all_modes],
            "P (OR)": [float(policy_share_out_abs.get(m, 0.0)) for m in all_modes_abs],
        }
    )
    # .sort_values(["Base (Berlin)", "Policy (Berlin)"], ascending=False)
    .reset_index(drop=True)
)

latex_table = mode_share_table.to_latex(
    index=False,
    column_format="lcccccc",
    float_format=lambda x: f"{x:.0f}",
)

# latex_table = (
#     latex_table.replace(r"\toprule", r"\hline", 1)
#     .replace(r"\midrule", r"\hline", 1)
#     .replace(r"\bottomrule", r"\hline", 1)
# )

print(latex_table)

In [ ]:
base_share = mode_share(base_trips)
policy_share = mode_share(policy_trips)

base_share_in = mode_share(base_trips_in_zone)
policy_share_in = mode_share(policy_trips_in_zone)

base_share_out = mode_share(base_trips_out_zone)
policy_share_out = mode_share(policy_trips_out_zone)

all_modes = sorted(
    set(base_share.index)
    .union(set(policy_share.index))
    .union(set(base_share_in.index))
    .union(set(policy_share_in.index))
    .union(set(base_share_out.index))
    .union(set(policy_share_out.index))
)

mode_share_table = (
    pd.DataFrame(
        {
            "Mode": all_modes,
            "B (Berlin)": [float(base_share.get(m, 0.0)) for m in all_modes],
            "B (IR)": [float(base_share_in.get(m, 0.0)) for m in all_modes],
            "B (Out)": [float(base_share_out.get(m, 0.0)) for m in all_modes],
            "P (Berlin)": [float(policy_share.get(m, 0.0)) - float(base_share.get(m, 0.0)) for m in all_modes],
            "P (IR)": [float(policy_share_in.get(m, 0.0)) - float(base_share_in.get(m, 0.0)) for m in all_modes],
            "P (Out)": [float(policy_share_out.get(m, 0.0)) - float(base_share_out.get(m, 0.0)) for m in all_modes],
        }
    )
    # .sort_values(["Base (Berlin)", "Policy (Berlin)"], ascending=False)
    .reset_index(drop=True)
)

latex_table = mode_share_table.to_latex(
    index=False,
    column_format="lcccccc",
    float_format=lambda x: f"{x:.1%}".replace("%", r"\%"),
)
latex_table = latex_table.replace(
    "Mode & B (Berlin) & B (IR) & B (Out) & P (Berlin) & P (IR) & P (Out)",
    " & \\multicolumn{3}{c}{Base modal share} & \\multicolumn{3}{c}{Pct. points change} \\\\\n"
    "Mode & Berlin & IR & OR & Berlin & IR & OR \\ ",
)

# latex_table = (
#     latex_table.replace(r"\toprule", r"\hline", 1)
#     .replace(r"\midrule", r"\hline", 1)
#     .replace(r"\bottomrule", r"\hline", 1)
# )

print(latex_table)


## Car Trips explicitly

In [ ]:
def analyze_car_ownership(trips) -> DataFrame:
    base_person_car_owner = (
        trips.assign(
            _has_car=trips["main_mode"].astype("string").fillna("").str.contains("car", regex=False)
        )
        .groupby("person", as_index=False)
        .agg(
            home_x=("home_x", "first"),
            home_y=("home_y", "first"),
            car_owner=("_has_car", "any"),
        )
    )
    return base_person_car_owner



In [ ]:
# group base_trips_in_zone by person, keep home_x/home_y, new column car owner: if any main mode contained car => True, if not => False
base_person_car_owner = analyze_car_ownership(base_trips_in_zone)
policy_person_car_owner = analyze_car_ownership(policy_trips_in_zone)

# join base_person_car_owner and policy_person_car_owner. Keep all columns.
person_car_owner = base_person_car_owner.merge(
    policy_person_car_owner,
    on="person",
    how="outer",
    suffixes=("_base", "_policy"),
)

person_car_owner


In [ ]:
import geopandas as gpd

pts = gpd.points_from_xy(person_car_owner["home_x_base"], person_car_owner["home_y_base"])
car_owner_gdf = gpd.GeoDataFrame(person_car_owner.copy(), geometry=pts, crs="EPSG:25832")

car_owner_gdf_3857 = car_owner_gdf.to_crs(epsg=3857)
zone_3857 = gdf.to_crs(epsg=3857)

fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharex=True, sharey=True)

policy_vals = [False, True]  # vertical (rows): False then True
base_vals = [False, True]  # horizontal (cols): False then True

for r, pol in enumerate(policy_vals):
    for c, base in enumerate(base_vals):
        ax = axes[r, c]

        zone_3857.plot(ax=ax, facecolor="green", edgecolor="black", linewidth=0.5, alpha=0.15)
        pl_car.plot(
            ax=ax,
            # column="vol_scaled",
            # cmap=cmap,
            # norm=norm,
            # linewidth=pl_car["_lw"],
            alpha=1.0,
            legend=True,
            legend_kwds={"label": "Traffic Volume in Traffic Volume in Passenger Car Equivalents", "shrink": 0.6},
        )

        subset = car_owner_gdf_3857[
            (car_owner_gdf_3857["car_owner_policy"] == pol) & (car_owner_gdf_3857["car_owner_base"] == base)
            ].copy()

        if len(subset) > 0:
            subset.plot(
                ax=ax,
                markersize=2,
                color="black",
                alpha=0.25,
            )

        try:
            ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=12)
        except Exception:
            pass

        ax.set_axis_off()
        ax.set_title(f"policy={pol}, base={base} (n={len(subset):,})", fontsize=11)

plt.tight_layout()
# plt.savefig("car-ownership-facets.png", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()


Interpretation: I don't find it obvious from this plot, who still uses the car. I expected the ones using the car to be those who are linging closer to the car streets. But more probably, the end point of the trip is the determining factor.